In [ ]:
#| default_exp support

In [ ]:
#| hide
from fastcore.test import *
from nbdev.showdoc import *
from tempfile import TemporaryDirectory

Check and install kernel support in a selected Python environment.

In [ ]:
#| export
import importlib.util, os, re, shutil, subprocess, sys, threading, time
from functools import cache
from pathlib import Path
from kunda.pythons import BUNDLE_ONLY, VENV_BINS, clean_env, nearest_python, strip_bundle

In [ ]:
#| export
HOST_PY = tuple(sys.version_info[:2])
INSPECTOR, INSPECTOR_TTL = ('dhrishti',), 90
KERNEL_PACKAGES = ('ipykernel', 'ipymini', 'kernmini', 'fastcore', 'microio')
#: ipykernel 7.0 to 7.3 can stop reading its shell for good when a request meets a reply (ipykernel#1529); 7.4 fixed it.
IPYKERNEL_FLOOR, MIN_PY = (7, 4), (3, 11)
_INSPECT, _locks, _locks_lock = {}, {}, threading.Lock()

`KERNEL_PACKAGES` lists the required distributions. `kernel_requirements` turns them into what an install asks for. `IPYKERNEL_FLOOR` is the first ipykernel without the shell stall, and `MIN_PY` the oldest Python kunda installs into.

In [ ]:
HOST_PY, INSPECTOR, KERNEL_PACKAGES, IPYKERNEL_FLOOR, MIN_PY

((3, 11),
 ('dhrishti',),
 ('ipykernel', 'ipymini', 'kernmini', 'fastcore', 'microio'))

In [ ]:
#| export
def work_dir(cwd=None):
    "`cwd` as a string, falling back to a writable directory when the process inherited `/` or a read-only bundle."
    if cwd: return str(cwd)
    here = os.getcwd()
    if not getattr(sys, 'frozen', False): return here
    return here if here != os.sep and os.access(here, os.W_OK) else str(Path.home())

`work_dir` returns a supplied directory as text. A missing directory uses the current working directory.

In [ ]:
work_dir(Path('/repo/notebooks'))

'/repo/notebooks'

In [ ]:
#| hide
here = os.getcwd()
try:
    sys.frozen = True; os.chdir(os.sep)
    test_eq(work_dir(), str(Path.home()))   # `/` is what a double-clicked app inherits
    test_eq(work_dir('/repo'), '/repo')     # a named directory is still kept
finally: os.chdir(here); del sys.frozen
test_eq(work_dir(), here)

In [ ]:
#| export
def support_paths():
    "The host's absolute, existing `sys.path` entries: where the inspector's finder looks for its own dependencies."
    return [p for p in sys.path if p and os.path.isabs(p) and os.path.exists(p)]

`support_paths` returns the existing absolute import paths of the host. Relative and missing paths are excluded. A kernel does not get these paths on its `sys.path`. Only the finder in `INSPECTOR_LOADER` searches them.

In [ ]:
#| export
def _lock_for(python):
    "One lock per interpreter: two installers in one venv is a race, and three surfaces offer it."
    with _locks_lock: return _locks.setdefault(str(python), threading.Lock())

Each interpreter path has one shared probe lock.

In [ ]:
test_is(_lock_for('/a/.venv/bin/python'), _lock_for('/a/.venv/bin/python'))
assert _lock_for('/a/.venv/bin/python') is not _lock_for('/b/.venv/bin/python')

In [ ]:
#| export
class _Failed:
    "What a command that never started looks like, so a probe reports instead of raising."
    returncode, stdout = 127, ''
    def __init__(self, err): self.stderr = err

def _run(args, timeout=180):
    "Run a command and return its result; missing interpreter and timeout are returned as answers, not raised."
    try:
        return subprocess.run([str(x) for x in args], env=clean_env(), text=True, stdin=subprocess.DEVNULL,
            encoding='utf-8', errors='replace', capture_output=True, timeout=timeout)
    except subprocess.TimeoutExpired: return _Failed(f'timed out after {timeout}s')
    except OSError as e: return _Failed(str(e))

`_run` always returns `returncode`, `stdout`, and `stderr`. Process-start failures become a non-zero result. Its stdin is `/dev/null`, so a probe never shares the host's terminal.

In [ ]:
p = _run(['/definitely/not/a/python', '-c', 'pass'])
p.returncode, p.stderr

(127, "[Errno 2] No such file or directory: '/definitely/not/a/python'")

In [ ]:
import os, pty, sys
m, s = pty.openpty(); os.write(m, b'x\n')   # a line waiting on the terminal, which only the parent may read
probe = "import os, select, sys; from kunda.support import _run; print(_run([sys.executable, '-c', 'import os; print(os.isatty(0))']).stdout.strip(), os.read(0, 9) if select.select([0], [], [], 1)[0] else b'')"
got = subprocess.run([sys.executable, '-c', probe], stdin=s, capture_output=True, text=True, timeout=60).stdout.split()
os.close(s); os.close(m)
test_eq(got, ['False', "b'x\\n'"])

In [ ]:
#| hide
p = _run([sys.executable, '-c', 'import time; time.sleep(5)'], timeout=1)
test_eq((p.returncode, p.stdout), (127, ''))
test_eq(p.stderr, 'timed out after 1s')

In [ ]:
#| export
def _last_line(text):
    "The line a traceback ends on. numpy's ABI failure wraps its own in eighteen lines of advice."
    return next((l for l in reversed((text or '').strip().splitlines()) if l.strip()), '').strip()

`_last_line` returns the last non-empty line, or an empty string.

In [ ]:
_last_line('Traceback (most recent call last):\n  File "<string>", line 1\nImportError: numpy ABI\n\n')

'ImportError: numpy ABI'

In [ ]:
#| hide
test_eq(_last_line(''), '')
test_eq(_last_line(None), '')
test_eq(_last_line('  only line  '), 'only line')

In [ ]:
#| export
#: The inspector only recognises values of these types. A kernel that has not loaded them holds none of their values.
VALUE_TYPES = ('numpy', 'pandas')

def _key(name): return re.sub(r'[-_.]+', '-', name).lower()

@cache
def inspector_modules(root=INSPECTOR[0], skip=VALUE_TYPES):
    "The top-level modules of `root`'s runtime dependencies, which a kernel may load from the host for `root` alone."
    from importlib.metadata import PackageNotFoundError, packages_distributions, requires
    seen, todo = set(), [root]
    while todo:
        if (k := _key(n := todo.pop())) in seen: continue
        seen.add(k)
        try: todo += [re.match(r'[\w.-]+', r)[0] for r in requires(n) or [] if 'extra ==' not in r]
        except PackageNotFoundError: pass
    return tuple(sorted(m for m, ds in packages_distributions().items()
                        if m != root and m not in skip and any(_key(d) in seen for d in ds)))

`inspector_modules` follows the runtime requirements of Dhrishti through the host's package metadata. It returns the import names of those packages. Dhrishti itself is not in the list, because the loader loads it by file path. `VALUE_TYPES` are not in the list. Dhrishti only checks whether a value is an ndarray or a DataFrame, and it reads numpy and pandas from `sys.modules` for that.

In [ ]:
inspector_modules()[:6]

In [ ]:
#| hide
if importlib.util.find_spec('dhrishti'):
    mods = inspector_modules()
    assert {'fastcore', 'fasthtml', 'safepyrun'} <= set(mods), mods
    assert not {'dhrishti', 'numpy', 'pandas', 'jupygate'} & set(mods), 'not Dhrishti, not value types, not what only kunda needs'
test_eq(inspector_modules('not_a_real_package_xyz'), ())

In [ ]:
#| export
INSPECTOR_LOADER = '''
def _kd_load():
	import sys, importlib.util
	from importlib.machinery import PathFinder
	sys.meta_path[:] = [f for f in sys.meta_path if type(f).__name__ != '_KdHost']
	if tuple(sys.version_info[:2]) != tuple({version!r}): return
	class _KdHost:
		"Find the inspector's own dependencies on the host's paths, and only for the inspector and what it loaded."
		names, served = frozenset({names!r}), set()
		def asker(self):
			f = sys._getframe(2)
			while f and f.f_globals.get('__name__', '').startswith(('importlib', '_frozen_importlib')): f = f.f_back
			return (f.f_globals.get('__name__') or '').partition('.')[0] if f else ''
		def find_spec(self, name, path=None, target=None):
			if path is not None or name not in self.names or (a := self.asker()) != 'dhrishti' and a not in self.served: return None
			if (s := PathFinder.find_spec(name, {support!r})): self.served.add(name)
			return s
	host = _KdHost()
	sys.meta_path.append(host)
	try:
		for _n in [n for n in sys.modules if n == 'dhrishti' or n.startswith('dhrishti.')]: del sys.modules[_n]
		_spec = importlib.util.spec_from_file_location('dhrishti', {dhrishti_init!r}, submodule_search_locations=[{dhrishti_dir!r}])
		_pkg = importlib.util.module_from_spec(_spec)
		sys.modules['dhrishti'] = _pkg
		_spec.loader.exec_module(_pkg)
	except Exception:
		sys.meta_path.remove(host)
		for _n in [n for n in sys.modules if n == 'dhrishti' or n.startswith('dhrishti.')]: del sys.modules[_n]
try: _kd_load()
finally: del _kd_load
'''

def inspector_loader():
    "Source that loads the host's Dhrishti into a kernel of the host's Python; '' when the host has no Dhrishti."
    spec = importlib.util.find_spec(INSPECTOR[0])
    if spec is None or not spec.origin: return ''
    init = str(Path(spec.origin).resolve())
    return INSPECTOR_LOADER.format(version=HOST_PY, names=inspector_modules(), support=support_paths(),
        dhrishti_init=init, dhrishti_dir=str(Path(init).parent))

`INSPECTOR_LOADER` loads the host's Dhrishti by file path. The kernel's `sys.path` does not change, so user code imports only what the kernel's own environment has. Dhrishti still needs its dependencies, for example fasthtml and safepyrun, and the environment often does not have them. The loader adds a finder at the end of `sys.meta_path` for these. The finder answers only when all of these are true:

1. The name is a top-level module in `inspector_modules`.
2. The environment does not have that module. The finder is last, so the environment's own copy is used first.
3. The import comes from Dhrishti, or from a module that the finder itself gave.

An import from user code goes to the finder too, and the finder refuses it. A module that Dhrishti already loaded stays in `sys.modules`, as every loaded module does. After the bootstrap, `import fasthtml` in user code can find Dhrishti's copy. A package that Dhrishti does not use, such as `vishalakshi`, is never found.

In a kernel of another Python minor version the loader does nothing. Then the bootstrap imports the environment's own Dhrishti, as before.

In [ ]:
import venv

In [ ]:
#| hide
if inspector_loader():
    bare = TemporaryDirectory(); venv.create(bare.name, with_pip=False, symlinks=True)   # same Python as the host, and no packages
    py = Path(bare.name)/'bin'/'python'
    check = inspector_loader() + """
import sys, dhrishti, dhrishti.core as c
print(dhrishti.__file__ == %r, c.is_gridable([1]), c.resolve_for([1]) is not None)
for m in ('jupygate', 'numpy', 'safepyrun'):
    try: __import__(m); print('found')
    except ModuleNotFoundError: print('refused')
print([p for p in %r if p in sys.path and 'site-packages' in p])
""" % (str(Path(importlib.util.find_spec('dhrishti').origin).resolve()), support_paths())
    got = _run([py, '-c', check], 120)
    test_eq(got.stdout.split('\n')[:4], ['True False True', 'refused', 'refused', 'refused'])
    test_eq(got.stdout.split('\n')[4], '[]')   # the host's site-packages never join the kernel's path

The test makes an empty venv of the host's Python. The loader loads Dhrishti there, and Dhrishti answers. User code cannot import `jupygate`, which the host has and Dhrishti does not use. It cannot import numpy either, because Dhrishti did not load it. `safepyrun` is refused too: Dhrishti imports it only when an agent runs, so it is not loaded yet.

In [ ]:
#| export
def _probe(python, module, pre=''):
    "Whether `module` imports in `python` after the source `pre` runs, with its version or the failure."
    p = _run([python, '-c', f'{pre}\nimport {module}; print(getattr({module}, "__version__", "installed"))'], 15)
    return {'available': p.returncode == 0,
        'version': p.stdout.strip() if p.returncode == 0 else '',
        'error': _last_line(p.stderr or p.stdout) if p.returncode else ''}

A module probe returns `available`, `version`, and `error`. `pre` runs first, in the same process: `inspector_support` passes `inspector_loader` there.

In [ ]:
tmp = TemporaryDirectory(); d = Path(tmp.name)
(d/'borrowed.py').write_text('__version__ = "1.2"')
_probe(sys.executable, 'borrowed', pre=f'import sys; sys.path.append({str(d)!r})')

{'available': True, 'version': '1.2', 'error': ''}

In [ ]:
#| hide
miss = _probe(sys.executable, 'borrowed')
test_eq((miss['available'], miss['version']), (False, ''))
assert 'ModuleNotFoundError' in miss['error']
test_eq(_probe(sys.executable, 'os')['version'], 'installed')

In [ ]:
#| export
def _vt(v):
    "The first two numbers of version string `v`, or None."
    try: return tuple(int(x) for x in str(v).split('.')[:2])
    except ValueError: return None

def _dist_version(python, name):
    "`name`'s version from the dist-info in the venv `python` belongs to, read off the disk; None when that cannot say."
    root = Path(python).parent.parent
    if not (root/'pyvenv.cfg').is_file(): return None
    found = {d.name[len(name)+1:-len('.dist-info')] for site in [*root.glob('lib*/python*/site-packages'), root/'Lib'/'site-packages']
             for d in site.glob(f'{name}-*.dist-info')}
    return found.pop() if len(found) == 1 else None    # two left by a broken install: only the interpreter knows which wins

def ipykernel_version(python=None):
    "The ipykernel version `python` launches, read from its metadata without importing it; '' when it has none."
    exe = str(Path(python).expanduser()) if python else sys.executable
    if os.path.abspath(exe) == os.path.abspath(sys.executable):
        from importlib.metadata import PackageNotFoundError, version
        try: return version('ipykernel')
        except PackageNotFoundError: return ''
    if (v := _dist_version(exe, 'ipykernel')): return v
    p = _run([exe, '-c', 'import importlib.metadata as m; print(m.version("ipykernel"))'], 15)
    return p.stdout.strip() if p.returncode == 0 else ''

def ipykernel_notice(python=None, version=None):
    "What a host shows for an ipykernel whose shell can stop answering, or None. `version` skips the read."
    v = ipykernel_version(python) if version is None else version
    if not (t := _vt(v)) or not (7, 0) <= t < IPYKERNEL_FLOOR: return None
    floor = '.'.join(map(str, IPYKERNEL_FLOOR))
    return {'kind': 'ipykernel_stall', 'python': str(python or ''), 'version': v, 'floor': floor,
        'message': f'This environment has ipykernel {t[0]}.{t[1]}, which can stop answering. Upgrade it to {floor} or later.'}

`ipykernel_version` reads the version from a venv's `dist-info` folder on disk, and starts the interpreter only when that folder cannot say. `ipykernel_notice` turns a version from 7.0 to 7.3 into a message a host can show.

In [ ]:
tmp_v = TemporaryDirectory(); venv = Path(tmp_v.name)
(venv/'pyvenv.cfg').write_text('home = /nowhere\n')
(venv/'lib/python3.12/site-packages/ipykernel-7.2.0.dist-info').mkdir(parents=True)
ipykernel_version(venv/'bin/python'), ipykernel_notice(version='7.2.0')['message']

In [ ]:
#| hide
test_eq(ipykernel_version(venv/'bin/python'), '7.2.0')   # no interpreter exists there, so this was read off the disk
test_eq(ipykernel_notice(venv/'bin/python')['version'], '7.2.0')
for v in ('7.4.0', '7.10.1', '6.29.5', '', 'installed'): test_is(ipykernel_notice(version=v), None)
test_eq(ipykernel_notice(version='7.0.1')['floor'], '7.4')
test_eq(ipykernel_version(sys.executable) == '', False)
test_is(ipykernel_notice(sys.executable), None)
(venv/'lib/python3.12/site-packages/ipykernel-7.4.1.dist-info').mkdir()
test_is(_dist_version(venv/'bin/python', 'ipykernel'), None)
test_eq(ipykernel_version(venv/'bin/python'), '')   # so it asks the interpreter, which is not there
tmp_v.cleanup()

In [ ]:
#| export
def kernel_support(python):
    "Importability of both supported kernel launchers in ``python``; the ipykernel entry carries `ipykernel_notice`."
    python = str(Path(python).expanduser())
    s = {m: _probe(python, m) for m in ('ipykernel', 'ipymini')}
    s['ipykernel']['notice'] = ipykernel_notice(python, s['ipykernel']['version']) if s['ipykernel']['available'] else None
    return s

`kernel_support` probes both kernel launchers in the selected interpreter. Its ipykernel entry also holds `notice`, which is None unless that ipykernel can stall.

In [ ]:
ok = kernel_support(sys.executable)['ipymini']
gone = kernel_support('/definitely/not/a/python')['ipymini']
ok, gone
assert kernel_support(sys.executable)['ipykernel']['notice'] is None

({'available': True, 'version': '0.1.21', 'error': ''},
 {'available': False,
  'version': '',
  'error': "[Errno 2] No such file or directory: '/definitely/not/a/python'"})

In [ ]:
#| export
def inspector_support(python, refresh=False):
    "Check whether `python` can import the live-variable inspector, returning what blocked it if not."
    python = str(Path(python).expanduser())
    hit = _INSPECT.get(python)
    if not refresh and hit and time.time() - hit[0] < INSPECTOR_TTL: return hit[1]
    state = _probe(python, 'dhrishti.serving', inspector_loader())
    _INSPECT[python] = (time.time(), state)
    return state

IMPORT_FAULTS = ('ModuleNotFoundError', 'ImportError', 'No module named')

def import_failure(err):
    "Whether bootstrap failed on an inspector import."
    return any(s in (err or '') for s in IMPORT_FAULTS)

`inspector_support` caches one result per interpreter for `INSPECTOR_TTL` seconds. `import_failure` recognizes a missing or broken inspector import.

In [ ]:
inspector_support(sys.executable), import_failure("ModuleNotFoundError: No module named 'dhrishti'")

{'available': False,
 'version': '',
 'error': "ModuleNotFoundError: No module named 'dhrishti'"}

In [ ]:
#| hide
s = inspector_support(sys.executable)
test_is(inspector_support(sys.executable), s)
_INSPECT[sys.executable] = (time.time() - INSPECTOR_TTL - 1, {'available': 'stale'})
test_eq(set(inspector_support(sys.executable)), {'available', 'version', 'error'})   # expired, asked again

test_eq(import_failure(None), False)
test_eq(import_failure(''), False)
numpy_abi = ('RuntimeError: module compiled against API version 0x10 but this version of numpy is 0xf\n'
             'Consider using a different interpreter, or reinstalling numpy, or ...\n'
             'ImportError: numpy.core.multiarray failed to import')
assert import_failure(numpy_abi)

In [ ]:
#| export
def installable(python):
    "Whether anything may install into `python`. Never this one, and never inside a signed bundle."
    if not python: return False
    p = os.path.abspath(str(python))
    if p == os.path.abspath(sys.executable): return False
    return '/Contents/Resources/' not in p and '/Contents/MacOS/' not in p

`installable` rejects a missing path, the current interpreter, and interpreters inside a macOS application bundle.

In [ ]:
installable(sys.executable), installable('/repo/.venv/bin/python')

(False, True)

In [ ]:
#| export
def as_installed(names=KERNEL_PACKAGES):
    "Pin `names` to the versions this process runs; leave unpinned what it doesn't have, so installs stay compatible with the host."
    from importlib.metadata import PackageNotFoundError, version
    out = []
    for name in names:
        try: out.append(f'{name}=={version(name)}')
        except (PackageNotFoundError, ValueError, OSError): out.append(name)
    return out

`as_installed` pins names to versions installed in the host process.

In [ ]:
as_installed(['fastcore', 'ipymini', 'not_a_real_package'])

['fastcore==2.2.19', 'ipymini==0.1.21', 'not_a_real_package']

In [ ]:
#| export
def kernel_requirements(names=KERNEL_PACKAGES):
    "What an install asks for: ipykernel at `IPYKERNEL_FLOOR` or later, the rest pinned by `as_installed`."
    names = list(names)
    floor = 'ipykernel>=' + '.'.join(map(str, IPYKERNEL_FLOOR))
    return [floor if n == 'ipykernel' else r for n, r in zip(names, as_installed(names))]

`kernel_requirements` asks for ipykernel by its floor rather than the host's exact version, so a project that pins a later ipykernel still installs, and one on 7.0 to 7.3 is raised past the stall.

In [ ]:
kernel_requirements(['ipykernel', 'fastcore'])

In [ ]:
#| hide
r = kernel_requirements(n for n in ('ipykernel', 'fastcore'))
test_eq(r[0], 'ipykernel>=7.4')
assert r[1].startswith('fastcore==')

In [ ]:
#| export
def _attempts(python, packages):
    "Installer commands to try, in order, for environments that intentionally lack pip."
    out = []
    if uv := shutil.which('uv'): out.append([uv, 'pip', 'install', '--python', python, *packages])
    out.append([python, '-m', 'pip', 'install', *packages])
    if not uv and _run([python, '-m', 'ensurepip', '--upgrade']).returncode == 0:
        out.insert(0, [python, '-m', 'pip', 'install', *packages])
    return out

`_attempts` tries `uv` when available and always includes `python -m pip install`.

In [ ]:
_attempts('/nowhere/bin/python', ['ipymini==0.1.21'])[-1]

['/nowhere/bin/python', '-m', 'pip', 'install', 'ipymini==0.1.21']

In [ ]:
#| hide
cmds = _attempts('/nowhere/bin/python', ['a', 'b'])
test_eq(cmds[-1], ['/nowhere/bin/python', '-m', 'pip', 'install', 'a', 'b'])
if shutil.which('uv'): test_eq(cmds[0][1:5], ['pip', 'install', '--python', '/nowhere/bin/python'])

`_log` records the command, return code, and bounded output.

In [ ]:
#| export
def _log(command, p): return {'command': command, 'returncode': p.returncode,'output': (p.stdout + '\n' + p.stderr).strip()[-8000:]}
def _detail(logs): return next((a['output'] for a in reversed(logs) if a['output']), 'no installer answered')

In [ ]:
#| hide
logs = [_log(['pip'], _Failed('')), _log(['uv'], _Failed('no interpreter'))]
test_eq(logs[0], {'command': ['pip'], 'returncode': 127, 'output': ''})
test_eq(_detail(logs), 'no interpreter')
test_eq(_detail(logs[:1]), 'no installer answered')
test_eq(_log(['pip'], _Failed('x' * 9000))['output'], 'x' * 8000)

In [ ]:
#| export
def _python_version(python):
    "`(major, minor)` of `python`, or None when it does not answer."
    p = _run([python, '-c', 'import sys; print(*sys.version_info[:2])'], 15)
    try: return tuple(map(int, p.stdout.split()))[:2] if p.returncode == 0 else None
    except ValueError: return None

def install_kernel_support(python, packages=None):
    "Install kernel packages via uv or pip; verify the import works after, so failures name the package not the installer."
    python = str(Path(python).expanduser().absolute())
    if (v := _python_version(python)) and v < MIN_PY: raise RuntimeError(
        f'{python} is Python {v[0]}.{v[1]}, and kernel support needs Python {MIN_PY[0]}.{MIN_PY[1]} or later. '
        'Choose another interpreter, or make the environment again with a newer Python.')
    packages, logs, unimportable = list(packages or kernel_requirements()), [], ''
    with _lock_for(python):
        for command in _attempts(python, packages):
            p = _run(command, 600)
            logs.append(_log(command, p))
            if p.returncode == 0:
                support = kernel_support(python)
                if all(x['available'] for x in support.values()): return {'ok': True, 'python': python, 'support': support, 'attempts': logs}
                unimportable = '; '.join(f"{name} installed but does not import ({x['error']})" for name, x in support.items() if not x['available'])
    raise RuntimeError(f'kernel support installation failed: {unimportable or _detail(logs)}')

`install_kernel_support` installs `kernel_requirements` and succeeds only when both launchers import. It refuses a Python older than `MIN_PY` before it installs anything.

In [ ]:
#| hide
# No installer can start against a path that is not an interpreter, so this installs nothing.
test_fail(install_kernel_support, args=['/nowhere/bin/python'], contains='kernel support installation failed')
old_py = d/'old_python'
old_py.write_text('#!/bin/sh\necho 3 10\n'); old_py.chmod(0o755)
test_eq(_python_version(old_py), (3, 10))
test_fail(install_kernel_support, args=[old_py], contains='Python 3.10, and kernel support needs Python 3.11')
test_eq(_python_version(sys.executable), HOST_PY)

In [ ]:
#| export
def make_venv(root, python=None):
    "Make `root/.venv` on `python` (this Python's minor by default) with kernel support; its interpreter. An existing venv is kept; a half-made one is removed."
    root = Path(root)
    target = root/'.venv'
    with _lock_for(target):
        if (py := nearest_python(root, root)): return py
        if target.exists(): raise FileExistsError(f'{target} exists and holds no interpreter; remove it or choose an interpreter')
        try:
            if uv := shutil.which('uv'):
                p = _run([uv, 'venv', target, '--python', python or '%d.%d' % HOST_PY], 600)
                if p.returncode: raise RuntimeError(f'uv venv failed: {_last_line(p.stderr or p.stdout)}')
            else:
                import venv
                venv.EnvBuilder(with_pip=True).create(target)
            install_kernel_support(py := target/VENV_BINS[os.name == 'nt'])
            return str(py)
        except BaseException:
            shutil.rmtree(target, ignore_errors=True)
            raise

`make_venv` gives a project its own `.venv`, so the project never borrows an environment from a parent folder or from the host. It uses `uv venv` on the host's Python minor when `uv` is on `PATH`, and the standard `venv` module when it is not. Then it installs kernel support. A venv that exists already is kept. A `.venv` folder with no interpreter is refused, because it may hold something the person wants. When a step fails, the new folder is removed, so the next try starts clean. One lock per folder stops two callers from making the same venv at once.

In [ ]:
#| hide
kept = d.resolve()/'kept'; (kept/'venv'/'bin').mkdir(parents=True); (kept/'venv'/'bin'/'python').write_text('')
test_eq(make_venv(kept), str(kept/'venv'/'bin'/'python'))     # an existing venv is kept, and nothing runs
odd = d/'odd'; (odd/'.venv').mkdir(parents=True); (odd/'.venv'/'notes.txt').write_text('mine')
test_fail(make_venv, args=[odd], contains='holds no interpreter')
assert (odd/'.venv'/'notes.txt').exists(), 'a folder that is not a venv is never removed'
if shutil.which('uv'):
    test_fail(make_venv, args=[d/'bad'], kwargs={'python': str(d/'no_python')}, contains='uv venv failed')
    assert not (d/'bad'/'.venv').exists(), 'a failed venv is removed'

In [ ]:
#| export
def install_inspector_support(python, packages=INSPECTOR):
    "Give `python` its own copy of what the inspector imports, then ask it again."
    python = str(Path(python).expanduser().absolute())
    logs = []
    with _lock_for(python):
        for command in _attempts(python, packages):
            p = _run(command, 600)
            logs.append(_log(command, p))
            if p.returncode != 0: continue
            state = inspector_support(python, refresh=True)
            if state['available']: return {'ok': True, 'python': python, 'inspector': state, 'attempts': logs}
            raise RuntimeError(f'{" and ".join(packages)} are installed, and the inspector still will not import: {state["error"]}')
    raise RuntimeError(f'could not install {" and ".join(packages)}: {_detail(logs)}')

`install_inspector_support` installs Dhrishti and probes it again.

In [ ]:
#| hide
tmp.cleanup()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()